# Bài 06 · Lan truyền ngược: tự viết lượt ngược cho mạng MNIST

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/06-lan-truyen-nguoc.ipynb)

Notebook đi kèm [Bài 06 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/) (reel và demo tương tác nằm ở trang bài học).
Mạng MNIST 784 → 128 → 64 → 10 của khóa học có hơn một trăm nghìn tham số. Sau mỗi lô ảnh, ta cần biết với *từng*
tham số: tăng nó lên một chút thì mất mát tăng hay giảm, và nhanh đến đâu? Ta tự viết bằng numpy thuật toán trả lời
câu hỏi đó cho mọi tham số cùng lúc, kiểm nó từng con số, rồi dùng nó huấn luyện mạng trên dữ liệu MNIST thật.

**Bạn sẽ làm:**
1. Chạy lượt xuôi và lượt ngược bằng tay trên đồ thị 7 tham số của demo, rồi kiểm từng gradient bằng cách nhích thử.
2. Viết một autograd tí hon (kiểu micrograd) để thấy quy tắc chuỗi được tự động hóa thế nào.
3. Viết lượt ngược dạng ma trận cho mạng 784 → 128 → 64 → 10 và kiểm cả 6 tensor tham số bằng sai phân hữu hạn.
4. Đếm cái giá: phép nhân–cộng (MAC) của lượt ngược so với lượt xuôi, và bộ nhớ để giữ các giá trị trung gian.
5. Huấn luyện mạng 5 lượt bằng chính lượt ngược của bạn, ra đúng con số của reel và demo.

**Cần biết trước:** [Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/)
(cross-entropy) và [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/)
(bước cập nhật $\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}$). ReLU là của
[Bài 02](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/), mạng nhiều lớp là của
[Bài 03](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Ký hiệu theo chương *Neural Computation* của [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html),
với quy ước vectơ hàng $z = xW + b$ như mọi bài trước.
Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong reel và demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. `sci()` in số rất nhỏ dạng $3{,}5 \cdot 10^{-6}$. Python vẫn *gõ* số thập phân bằng dấu chấm
(`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn vạch của mọi trục tuyến tính (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(vn_tick))


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 3,5·10⁻⁶."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{vn(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(vn(109386, 0), vn(0.1, 1), vn(-0.5399, 3), pct(0.9727), sci(3.5e-6))

## 2 · Dữ liệu MNIST và câu hỏi của bài

MNIST (LeCun và cộng sự, 1998) gồm 70.000 ảnh chữ số viết tay xám 28×28: 60.000 ảnh huấn luyện, 10.000 ảnh kiểm tra.
Ô dưới tải 4 tệp gốc từ một bản sao công khai và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.
Mỗi ảnh được trải thành một hàng 784 số trong $[0, 1]$, như ở Bài 01.

In [ ]:
# MNIST: giống hệt reel/perceptron/data.py
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)  # mỗi hàng là một ảnh
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)
print("huấn luyện:", Xtr.shape, " kiểm tra:", Xte.shape)

Mạng của khóa học là MLP của MLSysBook: $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$, rồi softmax và cross-entropy (Bài 04). Ô dưới định nghĩa lượt xuôi và khởi tạo mạng đúng như
`reel/mnist_mlp/train.py` (khởi tạo He, seed 0), rồi cho nó xem ảnh huấn luyện đầu tiên.

Lượt xuôi trả về hai thứ: logit $z$, và các **giá trị trung gian** $(x, h_1, h_2)$. Lúc dự đoán thì không cần giữ
chúng, nhưng lượt ngược sẽ cần (mục 6 và 8).

In [ ]:
SIZES = [784, 128, 64, 10]


def init(seed=0):
    """Khởi tạo He như train.py: W ~ N(0, 2/n_in), b = 0. Trả về cả rng: lượt huấn luyện dùng tiếp nó để xáo ảnh."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    return params, rng


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def forward(params, x):
    """Lượt xuôi cho một lô x (mỗi hàng một ảnh): trả về logit z và các giá trị trung gian lượt ngược cần."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h2 @ W3 + b3, (x, h1, h2)


def cross_entropy(z, y):
    p = softmax(z)
    return -np.log(p[np.arange(len(y)), y]).mean()


params0, _ = init(0)
n_params = sum(t.size for layer in params0 for t in layer)
z0, _ = forward(params0, Xtr[:1])
p0 = softmax(z0)[0]
print(f"{vn(n_params, 0)} tham số; ảnh đầu tiên là số {ytr[0]}")
print(f"mạng chưa học cho số {ytr[0]} xác suất {pct(p0[ytr[0]], 1)}  →  mất mát −ln p = {vn(-np.log(p0[ytr[0]]), 2)}")
untrained = cross_entropy(forward(params0, Xte)[0], yte)
print(
    f"mất mát trung bình trên 10.000 ảnh kiểm tra: {vn(untrained, 4)}  (đoán đều 10 chữ số: ln 10 = {vn(np.log(10), 4)})"
)
assert n_params == 109386 and abs(untrained - 2.4257) < 1e-3, "khác với reel và demo: hãy kiểm tra lại dữ liệu"

Hình dưới là ảnh đó (trái) và 10 xác suất mà mạng chưa học gán cho 10 chữ số (phải). Cột của đáp án đúng có màu xanh
và nét gạch, các cột khác màu xám.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 2.8), gridspec_kw={"width_ratios": [1, 3]})
a.imshow(train_img[0], cmap="gray_r")
a.set_title(f"ảnh huấn luyện #0: số {ytr[0]}", fontsize=10)
a.axis("off")
is_true = np.arange(10) == ytr[0]
b.bar(
    np.arange(10),
    p0,
    color=np.where(is_true, BLUE, GREY),
    hatch=["//" if t else "" for t in is_true],
    edgecolor="white",
)
b.set(xticks=range(10), xlabel="chữ số", ylabel="xác suất", title="Mạng chưa học: xác suất cho từng chữ số")
b.annotate("đáp án đúng", (ytr[0], p0[ytr[0]]), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
vn_axes(plt.gcf())
plt.show()

Mạng chưa học chưa phân biệt được chữ số nào: nó chỉ cho số 5 xác suất 11,2%. Mất mát trung bình 2,4257 vì vậy gần
với 2,3026 của việc đoán đều 10 chữ số. Để giảm mất mát, gradient descent (Bài 05) cần $\partial \mathcal{L} / \partial \theta$ cho **từng** tham số trong 109.386 tham số.
Lỗi này là “của ai”? Chính xác hơn: nếu nhích một tham số lên một chút thì mất mát đổi bao nhiêu? Đó là một độ nhạy
cục bộ, tại đúng giá trị hiện tại của tham số và với đúng lô ảnh này, chứ không phải lời phán xét tham số nào gây lỗi.

Ta bắt đầu từ một mạng nhỏ đến mức tính được bằng tay.

## 3 · Đồ thị tính toán: mỗi phép toán là một nút

Demo của bài dùng một mạng thật nhưng rất nhỏ: một đầu vào $x$, hai nơ-ron ReLU, một nơ-ron sigmoid ở đầu ra và
cross-entropy cho nhãn $y \in \{0, 1\}$ (Bài 04). Nó có 7 tham số $w_1, b_1, w_2, b_2, v_1, v_2, c$:

$$z_1 = w_1x + b_1, \qquad z_2 = w_2x + b_2, \qquad h_i = \mathrm{ReLU}(z_i)$$

$$z = v_1h_1 + v_2h_2 + c, \qquad \hat y = \sigma(z), \qquad \mathcal{L} = -\left[y \ln \hat y + (1 - y)\ln(1 - \hat y)\right]$$

Viết thành một **đồ thị tính toán**, mỗi phép toán là một nút. Lượt xuôi đi từ $x$ xuống $\mathcal{L}$ và giữ lại giá
trị ở mọi nút. Ô dưới chạy lượt xuôi với đúng các giá trị khởi đầu của demo ($x = 1{,}5$ và nhãn $y = 1$).

In [ ]:
theta = {"w1": 0.6, "b1": -0.3, "w2": -0.4, "b2": 0.9, "v1": -0.5, "v2": 0.8, "c": -0.1}  # giống demo
x_demo, y_demo = 1.5, 1


def softplus(a):
    return max(a, 0.0) + math.log1p(math.exp(-abs(a)))  # ln(1 + eᵃ), không bị tràn số


def forward_small(t, x, y):
    """Lượt xuôi của đồ thị nhỏ: trả về giá trị của mọi nút (lượt ngược sẽ cần chúng)."""
    v = {"x": x}
    v["z1"] = t["w1"] * x + t["b1"]
    v["z2"] = t["w2"] * x + t["b2"]
    v["h1"] = max(0.0, v["z1"])
    v["h2"] = max(0.0, v["z2"])
    v["z"] = t["v1"] * v["h1"] + t["v2"] * v["h2"] + t["c"]
    v["yh"] = 1 / (1 + math.exp(-v["z"]))
    v["L"] = y * softplus(-v["z"]) + (1 - y) * softplus(v["z"])  # = −[y ln ŷ + (1 − y) ln(1 − ŷ)], tính ổn định
    return v


NODE = {"x": "x", "z1": "z₁", "z2": "z₂", "h1": "h₁", "h2": "h₂", "z": "z", "yh": "ŷ", "L": "L"}
val = forward_small(theta, x_demo, y_demo)
print("  ·  ".join(f"{NODE[k]} = {vn(val[k], 3)}" for k in ["z1", "z2", "h1", "h2", "z", "yh", "L"]))
print(f"mạng cho {pct(val['yh'], 1)} rằng nhãn là 1, nên L = −ln ŷ = {vn(val['L'], 3)}")
assert abs(val["L"] - 0.7763437730407396) < 1e-12, "khác với demo: hãy kiểm tra lại các giá trị khởi đầu"

Cả hai nơ-ron ReLU đều mở ($z_1, z_2 > 0$). Mạng cho $\hat y = 0{,}460$, tức 46,0% rằng nhãn là 1, nên mất mát là
$-\ln 0{,}460 = 0{,}776$.

## 4 · Quy tắc chuỗi chạy ngược qua từng nút

Muốn biết $\partial \mathcal{L}/\partial w_1$, ta đi ngược từ $\mathcal{L}$ về $w_1$. Quy tắc chuỗi nói đạo hàm của
một hàm hợp là tích các đạo hàm thành phần:

$$\frac{\partial \mathcal{L}}{\partial w_1} = \frac{\partial \mathcal{L}}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial h_1}\cdot\frac{\partial h_1}{\partial z_1}\cdot\frac{\partial z_1}{\partial w_1}$$

Mỗi thừa số là một **đạo hàm cục bộ**: nút chỉ cần biết phép toán của chính nó và các giá trị đã lưu ở lượt xuôi.
Lượt ngược nhận gradient từ phía trên, nhân với đạo hàm cục bộ, rồi chuyển xuống nút bên dưới. Đồ thị này có ba kiểu
nút, mà CS231n gọi là các mẫu của dòng gradient:

- **Phép cộng** chép nguyên gradient cho mọi đầu vào của nó: $\partial \mathcal{L}/\partial c = \partial \mathcal{L}/\partial z$.
- **Phép nhân** hoán đổi hai đầu vào: $\partial \mathcal{L}/\partial v_1 = \partial \mathcal{L}/\partial z \cdot h_1$,
  còn $\partial \mathcal{L}/\partial h_1 = \partial \mathcal{L}/\partial z \cdot v_1$.
- **ReLU** là một cái cổng: nhân với 1 nếu $z_i > 0$, với 0 nếu không.

Sigmoid và cross-entropy đi liền nhau cho một kết quả gọn: $\partial \mathcal{L}/\partial z = \hat y - y$.

### 🤔 Dự đoán trước
$x$ đi vào cả $z_1$ lẫn $z_2$. Vậy $\partial \mathcal{L}/\partial x$ là gradient đi qua nhánh 1, qua nhánh 2,
hay tổng của hai nhánh?

In [ ]:
def backward_small(t, v, y):
    """Lượt ngược: mỗi dòng là gradient từ phía trên × một đạo hàm cục bộ."""
    g = {"L": 1.0}
    g["yh"] = -y / v["yh"] + (1 - y) / (1 - v["yh"])  # ∂L/∂ŷ
    g["z"] = g["yh"] * v["yh"] * (1 - v["yh"])  # × σ′(z) = ŷ(1 − ŷ); bằng đúng ŷ − y
    g["v1"], g["v2"], g["c"] = g["z"] * v["h1"], g["z"] * v["h2"], g["z"]  # nhân: hoán đổi; cộng: chép
    g["h1"], g["h2"] = g["z"] * t["v1"], g["z"] * t["v2"]
    g["z1"] = g["h1"] * (v["z1"] > 0)  # ReLU: cổng mở (× 1) hay đóng (× 0)
    g["z2"] = g["h2"] * (v["z2"] > 0)
    g["w1"], g["b1"] = g["z1"] * v["x"], g["z1"]
    g["w2"], g["b2"] = g["z2"] * v["x"], g["z2"]
    g["x"] = g["z1"] * t["w1"] + g["z2"] * t["w2"]  # x được dùng hai lần: gradient của hai nhánh cộng lại
    return g


grad = backward_small(theta, val, y_demo)
print(f"∂L/∂ŷ = {vn(grad['yh'], 3)}  ×  σ′(z) = {vn(val['yh'] * (1 - val['yh']), 3)}  →  ∂L/∂z = {vn(grad['z'], 4)}")
print(f"đúng bằng ŷ − y = {vn(val['yh'] - y_demo, 4)}")
print(
    f"∂L/∂x = {vn(grad['z1'] * theta['w1'], 3)} (nhánh 1) + {vn(grad['z2'] * theta['w2'], 3)} (nhánh 2) = {vn(grad['x'], 3)}"
)
print("gradient của 7 tham số:", "  ".join(f"{p} {vn(grad[p], 3)}" for p in theta))
DEMO_GRAD = {  # cùng các số mà demo trên trang bài học hiện ra
    "w1": 0.40493616341667427, "b1": 0.26995744227778284, "w2": -0.6478978614666788, "b2": -0.43193190764445255,
    "v1": -0.32394893073333936, "v2": -0.16197446536666968, "c": -0.5399148845555657,
}  # fmt: skip
assert all(abs(grad[p] - DEMO_GRAD[p]) < 1e-12 for p in theta), "khác với demo"

Đúng là tổng. $x$ được dùng hai lần, nên nhích $x$ làm đổi cả $z_1$ lẫn $z_2$, và ảnh hưởng của hai đường cộng lại.
Quy tắc chung: **một giá trị được dùng nhiều lần thì các gradient chảy về nó cộng lại**. Đó là lý do autograd viết
`grad += …` chứ không phải `grad = …` (mục 5). Ở mạng MNIST, mỗi điểm ảnh đi vào cả 128 nơ-ron của lớp 1, nên gradient
của nó là tổng của 128 đường.

Hình dưới vẽ cả đồ thị sau hai lượt. Mỗi ô ghi giá trị của nút (dòng trên, lượt xuôi) và gradient $\partial L/\partial(\cdot)$
của nó (dòng dưới, lượt ngược). Mũi tên liền màu xanh là lượt xuôi; mũi tên gạch màu cam là lượt ngược, kèm đạo hàm cục
bộ mà gradient được nhân với khi đi qua cạnh đó.

In [ ]:
def draw_graph(ax, t, v, g):
    pos = {"x": (0, 0), "z1": (1, 0.75), "z2": (1, -0.75), "h1": (2, 0.75), "h2": (2, -0.75), "z": (3, 0)}
    pos |= {"yh": (4, 0), "L": (5, 0)}
    pos = {k: (1.6 * c, r) for k, (c, r) in pos.items()}
    W, H = 0.98, 0.62
    edges = [
        ("x", "z1", t["w1"]),
        ("x", "z2", t["w2"]),
        ("z1", "h1", float(v["z1"] > 0)),
        ("z2", "h2", float(v["z2"] > 0)),
    ]
    edges += [("h1", "z", t["v1"]), ("h2", "z", t["v2"]), ("z", "yh", v["yh"] * (1 - v["yh"])), ("yh", "L", g["yh"])]
    for a, b, d in edges:
        (xa, ya), (xb, yb) = pos[a], pos[b]
        x0, x1 = xa + W / 2, xb - W / 2
        ax.annotate("", (x1, yb + 0.07), (x0, ya + 0.07), arrowprops={"arrowstyle": "-|>", "color": BLUE, "lw": 1.4})
        ax.annotate(
            "",
            (x0, ya - 0.07),
            (x1, yb - 0.07),
            arrowprops={"arrowstyle": "-|>", "color": ORANGE, "lw": 1.4, "ls": "--"},
        )
        side = 1 if ya + yb > 0 else -1  # nhãn nằm phía ngoài: cạnh trên ghi phía trên, cạnh dưới ghi phía dưới
        ax.text(
            (x0 + x1) / 2,
            (ya + yb) / 2 + side * 0.26,
            f"×{vn(d, 3)}",
            ha="center",
            va="center",
            fontsize=8,
            bbox={"fc": "white", "ec": "none", "pad": 0.5},
        )
    for k, (cx, cy) in pos.items():
        ax.add_patch(FancyBboxPatch((cx - W / 2, cy - H / 2), W, H, boxstyle="round,pad=0.02", fc="white", ec=GREY))
        ax.text(cx, cy + 0.14, f"{NODE[k]} = {vn(v[k], 3)}", ha="center", va="center", fontsize=9, weight="bold")
        ax.text(cx, cy - 0.15, f"∂L/∂{NODE[k]} = {vn(g[k], 3)}", ha="center", va="center", fontsize=8)
    ax.set(xlim=(-0.6, 5 * 1.6 + 0.6), ylim=(-1.25, 1.25))
    ax.axis("off")
    ax.legend(
        [Line2D([], [], color=BLUE, lw=1.6), Line2D([], [], color=ORANGE, lw=1.6, ls="--")],
        ["lượt xuôi: giá trị", "lượt ngược: gradient × đạo hàm cục bộ"],
        loc="upper right", frameon=False, fontsize=8,
    )  # fmt: skip


fig, ax = plt.subplots(figsize=(10, 3.6))
draw_graph(ax, theta, val, grad)
ax.set_title(f"Đồ thị tính toán của demo: x = {vn(x_demo, 1)}, y = {y_demo}", fontsize=11)
vn_axes(plt.gcf())
plt.show()

Đọc hình từ phải sang trái là đọc lượt ngược: $\partial L/\partial L = 1$, nhân $-2{,}174$ ra $\partial L/\partial \hat y$,
nhân $\sigma'(z) = 0{,}248$ ra $\partial L/\partial z = -0{,}540$, rồi tách làm hai nhánh. Nhánh trên nhân $v_1 = -0{,}5$
nên đổi dấu: $\partial L/\partial w_1 > 0$, tức tăng $w_1$ thì lỗi *tăng*. Hai cổng ReLU đều mở (× 1).

### Kiểm tra bằng cách nhích thử
Gradient nói: nếu tăng $w_1$ thêm $h$ thì $L$ đổi khoảng $\frac{\partial L}{\partial w_1} h$. Ta kiểm điều đó y như nút
“Nhích thử” của demo: tăng từng tham số thêm $h = 0{,}001$, chạy lại lượt xuôi, đo $\Delta L$.

In [ ]:
h = 1e-3
passes = 0
for p, value in theta.items():
    nudged = dict(theta, **{p: value + h})
    dL = forward_small(nudged, x_demo, y_demo)["L"] - val["L"]
    passes += 1
    print(f"{p:>2}: ΔL đo được {vn(dL, 7):>11}   ∂L/∂{p}·h = {vn(grad[p] * h, 7):>11}")
    assert abs(dL - grad[p] * h) < 1e-3 * abs(grad[p] * h)
print(
    f"đã tốn thêm {passes} lượt xuôi, mỗi tham số một lượt; lan truyền ngược cho cả {len(theta)} gradient chỉ với 1 lượt ngược"
)

Hai cột khớp nhau đến chữ số thứ ba có nghĩa, phần lệch còn lại là vì $h$ không nhỏ vô cùng (mục 7 làm kỹ hơn).
Nhích thử thì dễ viết và luôn đúng, nhưng tốn **một lượt xuôi cho mỗi tham số**: 7 lượt ở đây, 109.386 lượt cho
mạng MNIST. Lan truyền ngược cho tất cả gradient chỉ với một lượt xuôi và một lượt ngược.

### Một vòng hoàn chỉnh
Lan truyền ngược chỉ *tính* gradient. Việc *dùng* nó là của gradient descent (Bài 05):
$\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$, với $\eta = 0{,}5$ như demo.

In [ ]:
eta_demo = 0.5
theta_next = {p: theta[p] - eta_demo * grad[p] for p in theta}
L_next = forward_small(theta_next, x_demo, y_demo)["L"]
print("  ".join(f"{p} {vn(theta[p], 2)} → {vn(theta_next[p], 3)}" for p in theta))
print(f"mất mát: {vn(val['L'], 3)} → {vn(L_next, 3)}")
assert abs(L_next - 0.3137786399893179) < 1e-12, "khác với demo"

Một vòng forward → loss → backward → update đưa mất mát từ 0,776 xuống 0,314. Huấn luyện mạng MNIST là lặp đúng
vòng này hàng nghìn lần, chỉ khác là mỗi nút là một ma trận.

## 5 · Tự động hóa: một autograd tí hon

`backward_small` viết tay từng dòng cho đúng đồ thị này. Mỗi nút chỉ cần biết đạo hàm cục bộ của nó, nên máy có thể tự
làm phần còn lại: lúc tính xuôi, mỗi giá trị nhớ nó được tạo từ những giá trị nào và đạo hàm cục bộ là bao nhiêu; lúc
tính ngược, đi qua các nút theo thứ tự ngược lại và cộng dồn gradient. Đó là ý tưởng của
[micrograd](https://github.com/karpathy/micrograd) (Karpathy), và của autograd trong PyTorch hay JAX, ở quy mô tensor.

In [ ]:
class Value:
    """Một số vô hướng nhớ nó được tạo từ đâu, để tự chạy quy tắc chuỗi ngược lại (giống micrograd)."""

    def __init__(self, data, parents=(), local=()):
        self.data, self.grad = data, 0.0
        self.parents, self.local = parents, local  # local[i] = ∂(nút này)/∂(parents[i]), lưu ngay lúc tính xuôi

    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data + o.data, (self, o), (1.0, 1.0))  # phép cộng: chép gradient

    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data * o.data, (self, o), (o.data, self.data))  # phép nhân: hoán đổi

    def relu(self):
        return Value(max(0.0, self.data), (self,), (float(self.data > 0),))

    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data))
        return Value(s, (self,), (s * (1 - s),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def backward(self):
        order, seen = [], set()

        def visit(v):  # thứ tự tô-pô: một nút đứng sau mọi nút tạo ra nó
            if id(v) not in seen:
                seen.add(id(v))
                for p in v.parents:
                    visit(p)
                order.append(v)

        visit(self)
        self.grad = 1.0
        for v in reversed(order):  # đi ngược: khi tới v, mọi nút dùng v đã gửi xong gradient cho nó
            for p, d in zip(v.parents, v.local):
                p.grad += v.grad * d  # +=: một giá trị dùng nhiều lần thì gradient cộng lại
        return len(order)

Ô dưới dựng lại đúng đồ thị của demo bằng `Value`. Với nhãn $y = 1$, mất mát là $L = -\ln \hat y$. Ta không viết một dòng
đạo hàm nào, chỉ gọi `L.backward()`.

In [ ]:
P = {p: Value(v) for p, v in theta.items()}
xv = Value(x_demo)
h1v = (P["w1"] * xv + P["b1"]).relu()
h2v = (P["w2"] * xv + P["b2"]).relu()
yhv = (P["v1"] * h1v + P["v2"] * h2v + P["c"]).sigmoid()
L = yhv.log() * -1.0  # y = 1
n_nodes = L.backward()
print(f"L = {vn(L.data, 3)}; {n_nodes} nút trong đồ thị")
print("autograd:", "  ".join(f"{p} {vn(P[p].grad, 3)}" for p in theta), f"  x {vn(xv.grad, 3)}")
assert all(abs(P[p].grad - grad[p]) < 1e-12 for p in theta) and abs(xv.grad - grad["x"]) < 1e-12

Cùng 7 gradient (và $\partial L/\partial x$) với bản viết tay, chính xác đến sai số làm tròn. Hai chi tiết đáng nhớ:

- Mỗi nút giữ lại đạo hàm cục bộ (cũng tức là các giá trị trung gian) từ lượt xuôi cho đến khi lượt ngược đi qua.
  Đó là bộ nhớ mà huấn luyện phải trả thêm so với chỉ dự đoán (mục 8).
- `p.grad += …`: nếu viết `=` thì $x$ chỉ giữ gradient của nhánh được xử lý sau cùng. Đây là một lỗi rất hay gặp khi tự
  viết autograd.

Ý tưởng này có từ trước khi có mạng nơ-ron. Linnainmaa mô tả chế độ ngược của đạo hàm tự động trong luận văn năm 1970,
để đo sai số làm tròn lan qua một phép tính dài (Griewank, 2012). Werbos áp dụng nó cho mạng nơ-ron năm 1982. Bài báo
năm 1986 của Rumelhart, Hinton & Williams cho thấy nó giúp mạng nhiều lớp tự học được những biểu diễn hữu ích, và làm
nó trở nên phổ biến.

## 6 · Lượt ngược dạng ma trận cho mạng MNIST

Mạng 784 → 128 → 64 → 10 vẫn là một đồ thị như vậy, chỉ khác là mỗi nút giờ là một ma trận. Với một lô $B$ ảnh (mỗi hàng
của $X$ là một ảnh), đặt $a_1 = XW_1 + b_1$, $h_1 = \mathrm{ReLU}(a_1)$, $a_2 = h_1W_2 + b_2$, $h_2 = \mathrm{ReLU}(a_2)$,
$z = h_2W_3 + b_3$, và gọi $\mathcal{L}$ là cross-entropy trung bình của lô.

Đầu ra: Bài 04 cho đạo hàm của softmax + cross-entropy theo logit là $p - y$ ($y$ one-hot). Vì $\mathcal L$ là trung
bình của $B$ ảnh, ta chia thêm cho $B$:

$$\frac{\partial \mathcal L}{\partial z} = \frac{1}{B}(P - Y)$$

Sau đó mỗi lớp $z = hW + b$ lặp lại cùng ba công thức (phương trình 22–24 của MLSysBook), còn ReLU nhân với một mặt nạ:

$$\frac{\partial \mathcal L}{\partial W} = h^\top \frac{\partial \mathcal L}{\partial z}, \qquad \frac{\partial \mathcal L}{\partial b} = \mathbf{1}^\top \frac{\partial \mathcal L}{\partial z}, \qquad \frac{\partial \mathcal L}{\partial h} = \frac{\partial \mathcal L}{\partial z}\, W^\top, \qquad \frac{\partial \mathcal L}{\partial a} = \frac{\partial \mathcal L}{\partial h} \odot [a > 0]$$

Đây vẫn là phép nhân hoán đổi của mục 4, viết cho ma trận: gradient của $W$ dùng *đầu vào* $h$ đã lưu, còn gradient của
$h$ dùng *trọng số* $W$. $\mathbf 1^\top(\cdot)$ cộng các hàng lại: $b$ được dùng cho cả $B$ ảnh, nên gradient của $B$
ảnh cộng lại. Không cần thuộc chỗ đặt dấu chuyển vị: $\partial\mathcal L/\partial W$ phải có đúng hình dạng của $W$, và
chỉ có một cách nhân cho ra hình dạng đó (CS231n gọi đây là phân tích chiều).

Ô dưới viết lượt ngược đúng như `train.py`, chạy nó trên 64 ảnh huấn luyện đầu tiên và in hình dạng từng gradient.
Tham số `relu_gate` chỉ để mục 7 cố tình tạo ra một lỗi.

In [ ]:
def backward(params, cache, z, y, relu_gate=True):
    """Lượt ngược của mạng MNIST: gradient của cross-entropy trung bình theo cả 6 tensor tham số."""
    x, h1, h2 = cache  # các giá trị trung gian mà lượt xuôi đã giữ lại
    _, (W2, _), (W3, _) = params
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = p − y (Bài 04) …
    dz /= len(y)  # … chia cho B vì L là trung bình của lô
    da2 = dz @ W3.T  # ∂L/∂h₂
    if relu_gate:
        da2 = da2 * (h2 > 0)  # qua cổng ReLU: ∂L/∂a₂
    da1 = da2 @ W2.T  # ∂L/∂h₁
    if relu_gate:
        da1 = da1 * (h1 > 0)  # ∂L/∂a₁
    return [[x.T @ da1, da1.sum(0)], [h1.T @ da2, da2.sum(0)], [h2.T @ dz, dz.sum(0)]]


NAMES = ["W₁", "b₁", "W₂", "b₂", "W₃", "b₃"]


def tensors(params):
    return [t for layer in params for t in layer]


xb, yb = Xtr[:64], ytr[:64]
zb, cache = forward(params0, xb)
grads = backward(params0, cache, zb, yb)
for name, P, G in zip(NAMES, tensors(params0), tensors(grads)):
    print(f"{name}: tham số {P.shape!s:>10}   gradient {G.shape!s:>10}")
    assert P.shape == G.shape

Mỗi gradient có đúng hình dạng của tham số của nó: 109.386 con số, sau một lượt xuôi và một lượt ngược.

### Cộng qua mọi đường đi
Công thức của trang bài học viết cho một trọng số nối thẳng vào đầu ra. Với một trọng số của lớp 1, chẳng hạn
$W_1[i, j]$ nối điểm ảnh $i$ vào nơ-ron ẩn $j$, ảnh hưởng của nó đi tới $\mathcal L$ theo rất nhiều đường: qua mỗi ảnh
$n$ của lô, mỗi nơ-ron $m$ của lớp 2, mỗi logit $k$. Quy tắc chuỗi cho mỗi đường một tích các đạo hàm cục bộ, và gradient
là tổng của chúng:

$$\frac{\partial \mathcal L}{\partial W_1[i,j]} = \sum_{n}\sum_{m}\sum_{k} x_{ni}\,[a_{1,nj} > 0]\,W_2[j,m]\,[a_{2,nm} > 0]\,W_3[m,k]\,\frac{\partial \mathcal L}{\partial z_{nk}}$$

Ô dưới cộng đúng tổng đó bằng ba vòng lặp lồng nhau, cho trọng số lớp 1 có gradient lớn nhất, rồi so với lượt ngược.

In [ ]:
x0, h1, h2 = cache
dz = (softmax(zb) - np.eye(10)[yb]) / len(yb)
(W1, _), (W2, _), (W3, _) = params0
i, j = np.unravel_index(np.abs(grads[0][0]).argmax(), W1.shape)
total, n_paths, n_open = 0.0, 0, 0
for n in range(len(yb)):
    for m in range(64):
        for k in range(10):
            term = x0[n, i] * (h1[n, j] > 0) * W2[j, m] * (h2[n, m] > 0) * W3[m, k] * dz[n, k]
            n_paths += 1
            n_open += term != 0
            total += term
print(
    f"W₁[{i}, {j}]: tổng qua {vn(n_paths, 0)} đường đi (64 ảnh × 64 nơ-ron lớp 2 × 10 logit), {vn(n_open, 0)} đường khác 0"
)
print(f"vòng lặp: {vn(total, 6)}   lượt ngược: {vn(grads[0][0][i, j], 6)}")
assert np.isclose(total, grads[0][0][i, j], rtol=1e-12, atol=0)

Hai cách cho cùng một số. Đường nào đi qua một ảnh không có nét ở điểm ảnh $i$, hoặc qua một cổng ReLU đang đóng, thì
đóng góp đúng bằng 0. Liệt kê từng đường cho từng trọng số thì quá tốn: riêng một trọng số này đã có 40.960 đường.
Lượt ngược không liệt kê: nó tính $\partial\mathcal L/\partial h_2$ rồi $\partial\mathcal L/\partial h_1$ **một lần**, và
mọi trọng số phía trước dùng lại các kết quả đó. Phần chung của mọi đường được tính một lần, đó là chỗ tiết kiệm.

## 7 · Kiểm từng tensor bằng sai phân hữu hạn

Một lượt ngược viết tay có thể sai mà vẫn chạy, vẫn ra đúng hình dạng. Cách kiểm chuẩn là so nó với gradient *số*,
đo trực tiếp bằng cách nhích thử như ở mục 4, nhưng dùng sai phân trung tâm, chính xác hơn nhiều:

$$\frac{\partial \mathcal L}{\partial \theta_i} \approx \frac{\mathcal L(\theta_i + h) - \mathcal L(\theta_i - h)}{2h}$$

Ta so hai con số bằng **sai số tương đối** $|a - n| / \max(|a|, |n|)$, vì gradient có thể rất nhỏ hoặc rất lớn.
Theo CS231n, dưới $10^{-7}$ là tốt; với hàm có điểm gãy như ReLU, dưới $10^{-4}$ thường vẫn ổn; trên $10^{-2}$ thì
nhiều khả năng sai. CS231n còn dặn kiểm vài phần tử của **mọi** tensor, kể cả bias.

Ô dưới kiểm cả 6 tensor trên lô 64 ảnh ở mục 6. Tensor nhỏ (≤ 1.000 phần tử) được kiểm toàn bộ. Với $W_1$ và $W_2$,
ta kiểm 300 phần tử chọn ngẫu nhiên. Mỗi phần tử tốn hai lượt xuôi.

In [ ]:
def loss_of(params, x, y):
    return cross_entropy(forward(params, x)[0], y)


def numerical(params, x, y, t, idx, h=1e-5):
    """Gradient số của phần tử idx trong tensor thứ t: (L(θ + h) − L(θ − h)) / 2h."""
    P = tensors(params)[t]
    old = P[idx]
    P[idx] = old + h
    L_plus = loss_of(params, x, y)
    P[idx] = old - h
    L_minus = loss_of(params, x, y)
    P[idx] = old  # trả lại giá trị cũ
    return (L_plus - L_minus) / (2 * h)


def rel_err(a, n):
    d = max(abs(a), abs(n))
    return 0.0 if d == 0 else abs(a - n) / d


rng_check = np.random.default_rng(1)
checks = []  # với mỗi tensor: danh sách (chỉ số, gradient số)
for t, P in enumerate(tensors(params0)):
    if P.size <= 1000:
        idxs = list(np.ndindex(P.shape))
    else:
        idxs = [np.unravel_index(f, P.shape) for f in rng_check.choice(P.size, 300, replace=False)]
    checks.append([(i, numerical(params0, xb, yb, t, i)) for i in idxs])


def worst_errors(grads):
    return [max(rel_err(tensors(grads)[t][i], n) for i, n in chk) for t, chk in enumerate(checks)]


worst = worst_errors(grads)
for name, chk, e in zip(NAMES, checks, worst):
    print(f"{name}: kiểm {vn(len(chk), 0):>3} phần tử, sai số tương đối lớn nhất {sci(e)}")
print(f"tổng cộng {vn(sum(map(len, checks)), 0)} phần tử, {vn(2 * sum(map(len, checks)), 0)} lượt xuôi")
assert max(worst) < 1e-4, "lượt ngược không khớp gradient số"

Cả 6 tensor đều khớp, sai số lớn nhất cỡ $10^{-6}$ trở xuống. Phần lệch đến từ làm tròn số khi hai giá trị
$\mathcal L(\theta \pm h)$ gần như bằng nhau, nhất là ở những phần tử có gradient rất nhỏ.

### 🤔 Dự đoán trước
Một lỗi rất hay gặp: quên nhân với cổng ReLU trong lượt ngược (gọi `backward(..., relu_gate=False)`).
Tensor nào trong 6 tensor sẽ trượt kiểm tra: tất cả, chỉ những tensor gần đầu vào, hay chỉ những tensor gần đầu ra?

In [ ]:
variants = {
    "đúng": grads,
    "quên cổng ReLU": backward(params0, cache, zb, yb, relu_gate=False),
    "quên chia cho B": [[len(yb) * g for g in layer] for layer in grads],  # gradient của tổng, không phải trung bình
}
print(f"{'':>16}" + "".join(f"{n:>10}" for n in NAMES))
table = {}
for label, g in variants.items():
    table[label] = worst_errors(g)
    print(f"{label:>16}" + "".join(f"{sci(e) if e < 0.01 else vn(e, 2):>10}" for e in table[label]))
gate = table["quên cổng ReLU"]
assert max(gate[4:]) < 1e-4 and min(gate[:4]) > 1e-2
assert all(abs(e - 63 / 64) < 1e-3 for e in table["quên chia cho B"])

Quên cổng ReLU chỉ làm hỏng $W_1, b_1, W_2, b_2$. $W_3$ và $b_3$ vẫn qua kiểm tra vì gradient của chúng được tính
*trước* khi lượt ngược đi qua cổng ReLU nào. Lượt ngược đi từ đầu ra về đầu vào, nên một lỗi ở một nút chỉ làm hỏng
những gì nằm phía trước nó (gần đầu vào hơn). Nếu chỉ kiểm lớp cuối, ta sẽ không thấy lỗi này. Quên chia cho $B$ thì
mọi gradient lớn gấp 64 lần, sai số tương đối là $63/64 \approx 0{,}98$ ở cả 6 tensor.

### Chọn bước nhích $h$
$h$ to hay nhỏ thì tốt hơn? Ô dưới thử $h$ từ $10^{-1}$ đến $10^{-12}$ trên trọng số có gradient lớn nhất của $W_2$,
bằng float64 (mặc định của numpy) và bằng float32 (kiểu số mà GPU thường dùng). Nó cũng đếm số cổng ReLU của lớp 2 đổi
trạng thái giữa $\theta + h$ và $\theta - h$ trên 64 ảnh.

In [ ]:
k2 = np.unravel_index(np.abs(grads[1][0]).argmax(), W2.shape)  # trọng số có gradient lớn nhất của W₂
a2 = grads[1][0][k2]
params32 = [[t.astype(np.float32) for t in layer] for layer in params0]
xb32 = xb.astype(np.float32)


def gate_flips(params, x, t, idx, h):
    """Số cổng ReLU của lớp 2 (64 ảnh × 64 nơ-ron) có trạng thái khác nhau giữa θ + h và θ − h."""
    P = tensors(params)[t]
    old = P[idx]
    P[idx] = old + h
    plus = forward(params, x)[1][2] > 0
    P[idx] = old - h
    minus = forward(params, x)[1][2] > 0
    P[idx] = old
    return int((plus != minus).sum())


hs = 10.0 ** -np.arange(1, 12.5, 0.5)
e64 = np.array([rel_err(a2, numerical(params0, xb, yb, 2, k2, h)) for h in hs])
e32 = np.array([rel_err(a2, numerical(params32, xb32, yb, 2, k2, h)) for h in hs])
nflip = np.array([gate_flips(params0, xb, 2, k2, h) for h in hs])
print(f"W₂[{k2[0]}, {k2[1]}]: gradient từ lượt ngược {vn(a2, 6)}")
for h, a, b, f in zip(hs[::2], e64[::2], e32[::2], nflip[::2]):
    print(f"h = {sci(h, 0):>7}: sai số float64 {sci(a):>9}   float32 {sci(b):>9}   cổng ReLU đổi trạng thái: {f}")
best = e64.argmin()
print(f"tốt nhất: float64 {sci(e64[best])} ở h = {sci(hs[best])}; float32 không lần nào dưới {sci(e32.min())}")
i5 = np.abs(np.log10(hs) + 5).argmin()  # h = 10⁻⁵, bước mà mục 7 dùng
assert e64[i5] < 1e-8 and e32.min() > 1e-4 and nflip[0] > 0

Hình dưới vẽ cùng các số đó: trục ngang là $h$, trục dọc là sai số tương đối (cả hai theo thang log). Đường xanh, chấm
tròn là float64; đường cam gạch, chấm vuông là float32. Vùng xám là những $h$ mà ít nhất một cổng ReLU đổi trạng thái
giữa $\theta + h$ và $\theta - h$. Hai đường chấm ngang là ngưỡng $10^{-7}$ và $10^{-2}$ của CS231n.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.axvspan(hs[nflip > 0].min() / 10**0.25, hs.max() * 10**0.25, color=GREY, alpha=0.18, lw=0)
ax.text(hs.max() / 1.2, 3e-12, "cổng ReLU\nđổi trạng thái", ha="right", va="bottom", fontsize=9)
ax.loglog(hs, e64, "o-", color=BLUE, ms=5, label="float64")
ax.loglog(hs, e32, "s--", color=ORANGE, ms=5, label="float32")
for level, note in [(1e-7, "10⁻⁷: tốt"), (1e-2, "10⁻²: nhiều khả năng sai")]:
    ax.axhline(level, color=GREY, lw=1, ls=":")
    ax.text(hs.min(), level * 1.6, note, fontsize=8, color="#444444")
ax.set(
    xlabel="bước nhích h",
    ylabel="sai số tương đối",
    ylim=(1e-12, 3),
    title="Gradient số so với lượt ngược, một trọng số của W₂",
)
ax.legend(frameon=False, loc="center left")
vn_axes(plt.gcf())
plt.show()

Có ba vùng:

- **$h$ lớn** (vùng xám): trong khoảng $[\theta - h, \theta + h]$ có một cổng ReLU đóng hoặc mở, tức hàm mất mát có một
  điểm gãy ở đó. Sai phân đo độ dốc trung bình qua chỗ gãy, nên lệch cỡ $10^{-3}$ đến $10^{-2}$ dù lượt ngược hoàn
  toàn đúng.
- **$h$ vừa phải** (quanh $10^{-5}$): float64 khớp đến khoảng $10^{-10}$.
- **$h$ quá nhỏ:** $\mathcal L(\theta + h)$ và $\mathcal L(\theta - h)$ gần như bằng nhau, phép trừ mất gần hết chữ số có
  nghĩa, nên sai số tăng trở lại.

float32 chỉ giữ khoảng 7 chữ số có nghĩa (float64 khoảng 16), nên ở bước nào nó cũng chỉ tốt nhất là 9,3·10⁻³, sát
ngưỡng $10^{-2}$ mà CS231n coi là “nhiều khả năng sai”. Vì vậy kiểm gradient luôn chạy bằng float64, với $h$ cỡ
$10^{-5}$, trên ít ảnh (ít ảnh thì ít điểm gãy).

## 8 · Cái giá: phép tính và bộ nhớ

### 🤔 Dự đoán trước
Lượt xuôi của mạng tốn 109.184 phép nhân–cộng (MAC) cho mỗi ảnh (Bài 01). Lượt ngược cho ra 109.386 gradient.
Nó tốn bao nhiêu MAC: ít hơn lượt xuôi, khoảng bằng, khoảng gấp đôi, hay gấp hàng trăm nghìn lần?

In [ ]:
fwd = bwd = 0
print(f"{'lớp':>11}{'xuôi: hW':>11}{'ngược ∂W = hᵀ·∂z':>20}{'ngược ∂h = ∂z·Wᵀ':>20}")
for layer, (n_in, n_out) in enumerate(itertools.pairwise(SIZES), 1):
    mac = n_in * n_out  # MAC cho mỗi ảnh của một phép nhân với ma trận n_in × n_out
    mac_h = 0 if layer == 1 else mac  # lớp 1: không ai cần gradient theo điểm ảnh x
    fwd, bwd = fwd + mac, bwd + mac + mac_h
    print(f"{f'{n_in} → {n_out}':>11}{vn(mac, 0):>11}{vn(mac, 0):>20}{vn(mac_h, 0):>20}")
print(f"mỗi ảnh: xuôi {vn(fwd, 0)} MAC, ngược {vn(bwd, 0)} MAC = {vn(bwd / fwd, 2)} lần lượt xuôi")
print(f"một bước huấn luyện (xuôi + ngược) = {vn((fwd + bwd) / fwd, 2)} lần lượt xuôi")
nudge = n_params * fwd  # nhích thử một phía: một lượt xuôi cho mỗi tham số
print(
    f"nhích thử từng tham số: {vn(n_params, 0)} lượt xuôi = {vn(nudge, 0)} MAC, gấp {vn(nudge / bwd, 0)} lần lượt ngược"
)
assert (fwd, bwd) == (109184, 118016)

Lượt ngược chỉ tốn 1,08 lần lượt xuôi. Mỗi lớp cần hai phép nhân ma trận cùng cỡ với lượt xuôi, một cho $\partial W$ và
một cho $\partial h$, nên với lớp dense lượt ngược thường tốn cỡ gấp đôi (MLSysBook). Ở mạng này, lớp đầu chiếm 100.352
trong 109.184 MAC, và lớp đó không cần $\partial\mathcal L/\partial x$ (không ai học các điểm ảnh), nên tỉ lệ chỉ còn
1,08. Cộng phép cộng độ lệch và cổng ReLU vào cũng không đổi bao nhiêu: chúng chỉ tốn cỡ $B \cdot n$ phép tính mỗi lớp.

Điều quan trọng là tỉ lệ này là một **hằng số**: lượt ngược đắt cỡ lượt xuôi, dù mạng có bao nhiêu tham số. Nhích thử
từng tham số thì đắt gấp số tham số. Ô dưới đo thời gian thật trên máy bạn đang chạy (con số tùy máy; thường gần với tỉ lệ
MAC, nhưng không đúng bằng).

In [ ]:
def timeit(f, repeat=200):
    f()
    t0 = time.perf_counter()
    for _ in range(repeat):
        f()
    return (time.perf_counter() - t0) / repeat


t_f = timeit(lambda: forward(params0, xb))
t_b = timeit(lambda: backward(params0, cache, zb, yb))
print(f"lô 64 ảnh trên máy này: xuôi {vn(t_f * 1e3, 3)} ms, ngược {vn(t_b * 1e3, 3)} ms (gấp {vn(t_b / t_f, 1)} lần)")
print(f"nhích thử cả {vn(n_params, 0)} tham số: khoảng {vn(n_params * t_f, 0)} giây cho MỘT bước gradient descent")

### Bộ nhớ: cái giá thật của lượt ngược
Lượt ngược cần $x, h_1, h_2$ và $z$ của lượt xuôi (công thức $\partial W = h^\top \partial z$ dùng đầu vào $h$ của lớp,
mặt nạ ReLU dùng $h > 0$). Vậy khi huấn luyện, mọi giá trị trung gian của cả lô phải nằm trong bộ nhớ cho đến khi lượt
ngược đi qua lớp của nó. Khi chỉ dự đoán, tính xong một lớp là bỏ được đầu vào của lớp đó. Huấn luyện còn phải giữ
gradient (cùng cỡ với tham số) và trạng thái của bộ tối ưu: Adam (Bài 07) giữ thêm hai số cho mỗi tham số.

Ô dưới dựng lại bảng bộ nhớ đơn giản hóa của MLSysBook (Bảng 8: lô 32 ảnh, mỗi số 4 byte FP32, Adam) cho đúng mạng này.
Như sách, 1 kB = 1.000 byte.

In [ ]:
def ledger(B, sizes=SIZES, adam=True, nbytes=4):
    """Bảng bộ nhớ đơn giản hóa của MLSysBook: số byte của từng phần khi huấn luyện và khi chỉ dự đoán."""
    n = sum(a * b + b for a, b in itertools.pairwise(sizes))  # số tham số
    saved = B * sum(sizes)  # x, h₁, h₂, z của cả lô: giữ từ lượt xuôi đến khi lượt ngược đi qua
    peak = B * max(a + b for a, b in itertools.pairwise(sizes))  # dự đoán: chỉ cần đầu vào và đầu ra của lớp đang tính
    train = {"tham số": n, "giá trị trung gian": saved, "gradient": n, "trạng thái Adam": 2 * n * adam}
    infer = {"tham số": n, "giá trị trung gian": peak}
    return {k: v * nbytes for k, v in train.items()}, {k: v * nbytes for k, v in infer.items()}


mem_train, mem_infer = ledger(32)
print(f"giá trị trung gian phải giữ khi huấn luyện, lô 32: 32 × {sum(SIZES)} = {vn(32 * sum(SIZES), 0)} số")
for part, nb in mem_train.items():
    print(f"{part:>19}: huấn luyện {vn(nb / 1000, 1):>6} kB   dự đoán {vn(mem_infer.get(part, 0) / 1000, 1):>6} kB")
ratio = sum(mem_train.values()) / sum(mem_infer.values())
print(
    f"{'tổng':>19}: huấn luyện {vn(sum(mem_train.values()) / 1e6, 1)} MB, dự đoán {vn(sum(mem_infer.values()) / 1000, 0)} kB"
    f"  →  gấp {vn(ratio, 1)} lần"
)
assert 32 * sum(SIZES) == 31552 and round(ratio, 1) == 3.4, "khác với Bảng 8 của MLSysBook"

Huấn luyện tốn 3,4 lần bộ nhớ của việc chỉ dự đoán, đúng như Bảng 8 của MLSysBook. Hình dưới vẽ cùng bảng đó thành hai
thanh. Mỗi phần có một màu và một kiểu gạch riêng, và được ghi tên ngay trên thanh.

In [ ]:
parts = [
    ("tham số", BLUE, ""),
    ("giá trị trung gian", ORANGE, "//"),
    ("gradient", GREEN, "\\\\"),
    ("trạng thái Adam", GREY, ".."),
]
fig, ax = plt.subplots(figsize=(9, 2.4))
for row, mem in enumerate([mem_infer, mem_train]):
    left = 0
    for part, color, hatch in parts:
        kb = mem.get(part, 0) / 1000
        if kb:
            ax.barh(row, kb, left=left, color=color, hatch=hatch, edgecolor="white", height=0.6)
            ax.text(left + kb / 2, row, f"{part}\n{vn(kb, 0)} kB", ha="center", va="center", fontsize=8)
            left += kb
    ax.text(left + 15, row, f"{vn(left, 0)} kB", va="center", fontsize=9, weight="bold")
ax.set(yticks=[0, 1], yticklabels=["dự đoán", "huấn luyện"], xlabel="kB (FP32, lô 32 ảnh)", xlim=(0, 2150))
ax.set_title("Bộ nhớ của mạng 784 → 128 → 64 → 10: huấn luyện so với chỉ dự đoán", fontsize=11)
vn_axes(plt.gcf())
plt.show()

Lượt ngược không tốn nhiều phép tính hơn bao nhiêu; cái nó thật sự đòi là **bộ nhớ**. Với mạng nhỏ này, phần lớn bộ
nhớ là tham số, gradient và trạng thái Adam. Khi lô lớn, các giá trị trung gian chiếm nhiều nhất: chúng tăng theo cỡ
lô và theo *tổng* độ rộng của mọi lớp, trong khi dự đoán chỉ cần lớp đang tính (mục 10 thử điều này). Khi bộ nhớ
không đủ, người ta có thể bỏ bớt giá trị trung gian rồi tính lại chúng lúc lượt ngược cần (*activation checkpointing*,
Chen và cộng sự, 2016): đổi thêm phép tính lấy bộ nhớ.

## 9 · Huấn luyện bằng lượt ngược của chính bạn

Giờ ta ghép bốn bước forward → loss → backward → update thành vòng huấn luyện của `reel/mnist_mlp/train.py`: khởi tạo He
(seed 0), mini-batch SGD với $\eta = 0{,}1$, lô 64 ảnh, 5 lượt (epoch) qua 60.000 ảnh, xáo ảnh bằng chính `rng` của lúc
khởi tạo. Vòng lặp ghi mất mát trung bình sau mỗi 50 bước và độ chính xác kiểm tra sau mỗi lượt.

In [ ]:
def train(seed=0, epochs=5, eta=0.1, batch=64, relu_gate=True, log_every=50):
    params, rng = init(seed)
    history, test_acc, running, step = [], [], [], 0
    for _ in range(epochs):
        order = rng.permutation(len(Xtr))
        for i in range(0, len(order), batch):
            idx = order[i : i + batch]
            z, cache = forward(params, Xtr[idx])  # 1. lượt xuôi, giữ giá trị trung gian
            running.append(cross_entropy(z, ytr[idx]))  # 2. mất mát
            grads = backward(params, cache, z, ytr[idx], relu_gate)  # 3. lượt ngược
            for P, G in zip(tensors(params), tensors(grads)):  # 4. cập nhật: gradient descent (Bài 05)
                P -= eta * G
            step += 1
            if step % log_every == 0:
                history.append(np.mean(running))
                running = []
        test_acc.append((forward(params, Xte)[0].argmax(1) == yte).mean())
    return params, np.array(history), np.array(test_acc)


t0 = time.perf_counter()
params, history, test_acc = train()
print(f"5 lượt, {vn(5 * math.ceil(len(Xtr) / 64), 0)} bước cập nhật trong {vn(time.perf_counter() - t0, 1)} giây")
print("độ chính xác kiểm tra sau mỗi lượt:", " → ".join(pct(a) for a in test_acc))
z_te = forward(params, Xte)[0]
test_loss = cross_entropy(z_te, yte)
train_acc = (forward(params, Xtr)[0].argmax(1) == ytr).mean()
print(f"mất mát huấn luyện: {vn(history[0], 4)} (50 bước đầu) → {vn(history[-1], 4)} (50 bước cuối)")
print(f"mất mát kiểm tra {vn(test_loss, 4)}  ·  đúng {pct(train_acc)} ảnh huấn luyện")
REF = [0.9427, 0.9633, 0.9688, 0.9706, 0.9727]  # epochLog của mnist-mlp.json (reel/mnist_mlp/train.py)
assert all(abs(a - r) < 5e-4 for a, r in zip(test_acc, REF)), "khác với reel và demo: hãy kiểm tra lại dữ liệu"
assert abs(test_loss - 0.0865) < 5e-4 and abs(train_acc - 0.9828) < 5e-4
assert abs(history[0] - 1.2031) < 1e-3 and abs(history[-1] - 0.059) < 1e-3

Sau 5 lượt, mạng đúng **97,27%** trên 10.000 ảnh kiểm tra, mất mát 0,0865: đúng con số của reel và demo Bài 04, vì đây
là cùng một phép tính với cùng seed. Ô dưới so cả 109.386 trọng số với bản tham chiếu mà `train.py` tạo ra. Trên máy dựng
notebook này, chúng trùng nhau từng bit. Trên CPU khác, thư viện BLAS có thể làm tròn khác ở chữ số cuối cùng: dòng dưới
khi đó in `False`, còn mọi con số đã làm tròn ở trên vẫn như cũ.

In [ ]:
md5 = hashlib.md5(b"".join(t.tobytes() for t in tensors(params))).hexdigest()
print("trùng từng bit với trọng số của train.py:", md5 == "e61017ca6e9fe52bc61dd8079ac77cd9")

Hình dưới là mất mát huấn luyện (trung bình mỗi 50 bước) theo số bước cập nhật. Các vạch chấm đứng đánh dấu cuối mỗi
lượt, ghi độ chính xác kiểm tra lúc đó.

In [ ]:
steps = 50 * np.arange(1, len(history) + 1)
per_epoch = math.ceil(len(Xtr) / 64)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(steps, history, color=BLUE, lw=2)
for e, a in enumerate(test_acc, 1):
    ax.axvline(e * per_epoch, color=GREY, lw=1, ls=":")
    ax.text(e * per_epoch - 40, 1.15, f"lượt {e}\n{pct(a)}", ha="right", va="top", fontsize=8)
ax.set(xlabel="bước cập nhật (lô 64 ảnh)", ylabel="mất mát huấn luyện", ylim=(0, 1.25), xlim=(0, 5 * per_epoch + 50))
ax.set_title("Huấn luyện bằng lượt ngược tự viết (η = 0,1, seed 0)", fontsize=11)
vn_axes(plt.gcf())
plt.show()

Mất mát rơi nhanh trong vài trăm bước đầu (từ 1,2031 ở 50 bước đầu), rồi giảm chậm dần xuống 0,059. Mỗi điểm trên đường
cong là trung bình của 50 lô, nên vẫn còn gợn: mỗi lô cho một gradient hơi khác nhau.

### 🤔 Dự đoán trước
Mỗi hàng của $W_1$ ứng với một điểm ảnh: hàng $i$ chứa 128 trọng số nối điểm ảnh $i$ vào lớp ẩn đầu tiên. Sau 5 lượt
và 4.690 bước cập nhật, có hàng nào vẫn **đúng bằng** giá trị lúc khởi tạo không? Nếu có, bao nhiêu hàng, và vì sao?

In [ ]:
params_init, _ = init(0)
unchanged = (params[0][0] == params_init[0][0]).all(axis=1)  # hàng nào của W₁ không đổi một bit nào
blank = Xtr.max(axis=0) == 0  # điểm ảnh trắng trong cả 60.000 ảnh huấn luyện
print(
    f"{unchanged.sum()} hàng của W₁ không đổi; {blank.sum()} điểm ảnh trắng trong mọi ảnh huấn luyện; trùng nhau: {np.array_equal(unchanged, blank)}"
)
one = backward(params_init, *forward(params_init, Xtr[:1])[::-1], ytr[:1])[0][0]
print(
    f"với riêng ảnh #0: {(Xtr[0] == 0).sum()} trong 784 hàng của ∂L/∂W₁ bằng 0 (điểm ảnh không có nét), "
    f"và {(one == 0).all(axis=0).sum()} trong 128 cột bằng 0 (nơ-ron có cổng ReLU đóng)"
)
assert unchanged.sum() == 67 and np.array_equal(unchanged, blank)

Đúng 67 hàng không đổi một bit nào, và đó chính là 67 điểm ảnh trắng trong *mọi* ảnh huấn luyện (chủ yếu ở các góc).
Gradient của $W_1$ là $x^\top\,\partial\mathcal L/\partial a_1$, nên hàng $i$ luôn nhân với $x_i$: điểm ảnh không bao giờ
có nét thì gradient của cả hàng luôn bằng đúng 0, và gradient descent không bao giờ chạm vào nó. Đây là cùng một lý do
khiến 297 trọng số của perceptron ở Bài 01 vẫn bằng 0.

Với một ảnh, gradient còn thưa hơn nhiều: hàng nào ứng với điểm ảnh trắng của ảnh đó, cột nào ứng với nơ-ron có cổng
ReLU đóng, đều bằng 0. Lan truyền ngược chỉ chia trách nhiệm cho những gì thật sự đã tham gia vào lượt xuôi.

### Một lỗi không làm hỏng gì, nhưng làm chậm tất cả
Ở mục 7, quên cổng ReLU làm sai 4 trong 6 tensor. Nhưng nó không làm chương trình báo lỗi. Ô dưới huấn luyện 1 lượt
với lượt ngược đúng và với lượt ngược thiếu cổng ReLU, mỗi bản 3 seed.

In [ ]:
for gate in [True, False]:
    accs = [train(seed, epochs=1, relu_gate=gate)[2][-1] for seed in range(3)]
    print(
        f"{'lượt ngược đúng' if gate else 'quên cổng ReLU':>16}: sau 1 lượt đúng {pct(min(accs))} đến {pct(max(accs))} (3 seed)"
    )
    if gate:
        good = accs
assert min(good) > max(accs)

Mạng với gradient sai *vẫn học*: nó vẫn đi xuống dốc theo một hướng gần đúng. Nhưng sau 1 lượt nó kém lượt ngược đúng
khoảng 3 điểm phần trăm, xa ngoài khoảng dao động giữa các seed. Không có thông báo lỗi nào; chỉ có một mô hình tệ hơn
một cách khó hiểu. Đó là lý do phải kiểm gradient từng tensor mỗi khi tự viết lượt ngược: nhìn đường mất mát đi xuống là
không đủ. Karpathy gọi lan truyền ngược là một “trừu tượng rò rỉ” (*leaky abstraction*): ngay cả khi dùng PyTorch, hiểu nó
giúp nhận ra vì sao mạng học kém.

## 10 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Bộ nhớ theo cỡ lô và cỡ mạng
Hàm `ledger()` của mục 8 nhận cỡ lô `B` và các lớp `sizes`. Thử lô lớn hơn, và một mạng lớn hơn (4 lớp ẩn 512):

In [ ]:
for B, sizes in [(32, SIZES), (256, SIZES), (4096, SIZES), (256, [784, 512, 512, 512, 512, 10])]:
    tr, inf = ledger(B, sizes)
    act = tr["giá trị trung gian"] / sum(tr.values())
    print(
        f"lô {B:>4}, mạng {' → '.join(map(str, sizes)):>28}: huấn luyện {vn(sum(tr.values()) / 1e6, 1):>5} MB, "
        f"dự đoán {vn(sum(inf.values()) / 1e6, 2):>5} MB (gấp {vn(sum(tr.values()) / sum(inf.values()), 1):>4} lần); "
        f"giá trị trung gian chiếm {pct(act, 0)} bộ nhớ huấn luyện"
    )

Khi lô lớn lên, giá trị trung gian từ 7% trở thành 90% bộ nhớ huấn luyện. Tỉ lệ huấn luyện so với dự đoán lại *giảm*
(3,4 → 2,0 → 1,2 lần), vì dự đoán cũng phải chứa cả lô, dù chỉ cho lớp đang tính. Với mạng 4 lớp ẩn 512, tham số,
gradient và trạng thái Adam lại chiếm phần lớn. Con số 3,4 lần vì vậy không phải hằng số: nó tùy cỡ lô, cỡ mạng và bộ
tối ưu, như MLSysBook lưu ý.

### Gradient của một lô so với của cả tập
Gradient descent ở Bài 05 dùng gradient của một lô 64 ảnh thay cho gradient của cả 60.000 ảnh. Lượt ngược tính được cả
hai. Ô dưới đo độ tương đồng cosin giữa chúng (1 là cùng hướng) cho 20 lô ngẫu nhiên, lúc mạng mới khởi tạo.

In [ ]:
def flat(g):
    return np.concatenate([t.ravel() for t in tensors(g)])


z_all, cache_all = forward(params_init, Xtr)
g_full = flat(backward(params_init, cache_all, z_all, ytr))
rng_b = np.random.default_rng(5)
cos = []
for _ in range(20):
    idx = rng_b.choice(len(Xtr), 64, replace=False)
    zb_, cb_ = forward(params_init, Xtr[idx])
    g = flat(backward(params_init, cb_, zb_, ytr[idx]))
    cos.append(g @ g_full / np.linalg.norm(g) / np.linalg.norm(g_full))
print(f"cosin giữa gradient của lô 64 ảnh và của cả 60.000 ảnh: {vn(min(cos), 2)} đến {vn(max(cos), 2)} (20 lô)")
nz = np.abs(g_full[g_full != 0])
lo, hi = np.percentile(nz, [1, 99])
print(f"độ lớn |∂L/∂θ| khác 0: từ {sci(lo)} (phân vị 1%) đến {sci(hi)} (phân vị 99%), chênh {sci(hi / lo)} lần")

Gradient của một lô chỉ *gần* đúng hướng của gradient thật, và các tham số nhận gradient có độ lớn chênh nhau hàng
trăm nghìn lần. Cùng một $\eta$ cho mọi tham số vì vậy là một lựa chọn thô. Đó là chỗ để cải tiến bước cập nhật, không
phải lượt ngược.

### Seed và cỡ lô
Thử các seed khác, hoặc lô nhỏ hơn: lượt ngược không đổi, chỉ có số bước và độ nhiễu của gradient thay đổi.

In [ ]:
for seed, batch in [(1, 64), (2, 64), (0, 16)]:
    t0 = time.perf_counter()
    acc = train(seed, epochs=1, batch=batch)[2][-1]
    print(f"seed {seed}, lô {batch:>2}: sau 1 lượt đúng {pct(acc)} ({vn(time.perf_counter() - t0, 1)} giây)")

Seed 1 và 2 nằm trong khoảng 94,17–94,86% của mục 9. Lô 16 ảnh cho 4 lần nhiều bước cập nhật hơn trong một lượt, nên
sau 1 lượt đã đạt 96,37%, đổi lại là gọi lượt ngược nhiều gấp 4 lần và gradient của mỗi bước nhiễu hơn.

## 11 · Giới hạn và bước tiếp theo

Lan truyền ngược đã làm xong việc của nó: gradient đúng (đã kiểm từng tensor) cho cả 109.386 tham số với giá xấp xỉ một
lượt xuôi. Phần còn thô là **bước cập nhật**: SGD trừ cùng một $\eta \cdot \partial\mathcal L/\partial\theta$ cho mọi tham
số, dù gradient của lô thì nhiễu và các tham số có gradient lớn nhỏ chênh nhau rất xa. Bài 07 thay bước đó bằng các bộ tối
ưu nhớ lịch sử gradient: momentum làm mượt hướng đi, Adam chia gradient của từng tham số cho độ lớn quen thuộc của nó.
Cái giá là thêm bộ nhớ: đúng hai số cho mỗi tham số, phần “trạng thái Adam” trong bảng của mục 8.

Một giới hạn nữa: lượt ngược đi qua mọi lớp, nên gradient của lớp đầu là tích của rất nhiều đạo hàm cục bộ. Với mạng sâu,
tích đó có thể co về 0 hoặc nổ tung: [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/).

## Tóm tắt
- Lượt xuôi tính và **giữ** các giá trị trung gian; lượt ngược đi từ $\mathcal L$ về, nhân gradient phía trên với đạo hàm
  cục bộ của từng nút. Demo 7 tham số: $\mathcal L$ = 0,776, và sau một bước cập nhật còn 0,314.
- Phép cộng chép gradient, phép nhân hoán đổi, ReLU là cổng; một giá trị dùng nhiều lần thì gradient **cộng lại**.
- Dạng ma trận: $\partial W = h^\top \partial z$, $\partial b = \mathbf 1^\top \partial z$, $\partial h = \partial z\, W^\top$,
  bắt đầu từ $\partial z = (P - Y)/B$.
- Kiểm bằng sai phân trung tâm, float64, $h$ cỡ $10^{-5}$, mọi tensor: sai số lớn nhất cỡ $10^{-6}$. Quên cổng ReLU vẫn
  qua kiểm tra ở $W_3$, $b_3$, và mạng vẫn học, chỉ kém khoảng 3 điểm phần trăm sau 1 lượt.
- Lượt ngược tốn 118.016 MAC so với 109.184 của lượt xuôi (1,08 lần); nhích thử thì cần 109.386 lượt xuôi. Cái giá thật
  là bộ nhớ: lô 32 ảnh, huấn luyện cần 3,4 lần bộ nhớ dự đoán.
- Huấn luyện bằng lượt ngược tự viết: 97,27% trên tập kiểm tra sau 5 lượt, đúng như reel và demo; 67 hàng của $W_1$ (các
  điểm ảnh luôn trắng) không bao giờ đổi.

## Câu hỏi ôn tập
1. Vì sao lượt xuôi phải giữ lại $h_1$ và $h_2$ khi huấn luyện, nhưng không cần khi chỉ dự đoán?
2. Ở demo, $x$ đi vào cả $z_1$ lẫn $z_2$. Vì sao $\partial L/\partial x$ là tổng của hai nhánh?
3. Gradient của $W_3$ có cần mặt nạ ReLU không? Điều đó giải thích kết quả nào ở mục 7?
4. Vì sao kiểm gradient bằng float32 hay với $h = 0{,}1$ có thể báo sai, dù lượt ngược đúng?
5. Vì sao lượt ngược của mạng này chỉ tốn khoảng 1,08 lần lượt xuôi, chứ không phải gấp đôi?
6. Sau 5 lượt, 67 hàng của $W_1$ vẫn đúng bằng giá trị khởi tạo. Vì sao?

<details><summary>Gợi ý đáp án</summary>

1. $\partial W_2 = h_1^\top \partial a_2$ và mặt nạ ReLU $[h > 0]$ đều cần chúng. Khi dự đoán, tính xong một lớp là bỏ
   được đầu vào của lớp đó.
2. Nhích $x$ làm đổi cả hai nhánh, và ảnh hưởng của mỗi đường đi tới $L$ cộng lại (quy tắc chuỗi nhiều biến).
3. Không: $\partial W_3 = h_2^\top \partial z$ được tính trước khi đi qua cổng ReLU nào. Vì vậy quên cổng ReLU vẫn để
   $W_3$ và $b_3$ qua kiểm tra; chỉ các tensor phía trước cổng bị sai.
4. Với $h$ lớn, khoảng $[\theta - h, \theta + h]$ có thể chứa điểm gãy của ReLU. float32 chỉ có khoảng 7 chữ số có nghĩa,
   nên phép trừ $\mathcal L(\theta + h) - \mathcal L(\theta - h)$ mất gần hết độ chính xác.
5. Mỗi lớp cần $\partial W$ và $\partial h$, mỗi thứ tốn bằng lượt xuôi của lớp đó. Nhưng lớp 1 (100.352 trong 109.184 MAC)
   không cần $\partial x$, vì không ai học các điểm ảnh.
6. Hàng $i$ của $\partial W_1 = x^\top \partial a_1$ luôn nhân với $x_i$; 67 điểm ảnh đó trắng trong cả 60.000 ảnh huấn
   luyện, nên gradient của chúng luôn bằng đúng 0.
</details>

## Tìm hiểu thêm
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): công thức lượt ngược
  (phương trình 22–24), ví dụ tính tay và bảng bộ nhớ huấn luyện so với dự đoán (Bảng 8) của bài này.
- D. Rumelhart, G. Hinton & R. Williams (1986), [Learning representations by back-propagating errors](https://doi.org/10.1038/323533a0).
- A. Griewank (2012), [Who Invented the Reverse Mode of Differentiation?](https://doi.org/10.4171/dms/6/38): Linnainmaa 1970.
- P. Werbos (1982), [Applications of advances in nonlinear sensitivity analysis](https://doi.org/10.1007/BFb0006203).
- CS231n, [Backpropagation, Intuitions](https://cs231n.github.io/optimization-2/) (các mẫu của dòng gradient, phân tích
  chiều) và [Gradient checks](https://cs231n.github.io/neural-networks-3/) (sai số tương đối, float64, điểm gãy).
- A. Karpathy, [micrograd](https://github.com/karpathy/micrograd) và bài giảng
  [Becoming a Backprop Ninja](https://youtu.be/q8SA3rM6ckI): tự viết lượt ngược cho cả một MLP.
- C. Olah (2015), [Calculus on Computational Graphs: Backpropagation](https://colah.github.io/posts/2015-08-Backprop/).
- T. Chen và cộng sự (2016), [Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174):
  activation checkpointing.

**Bài tiếp theo:** [Bài 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/): giữ nguyên
lượt ngược, cải tiến bước cập nhật.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")